# Anime Recommendation System - Collaborative Filtering DAE

This notebook builds a high-level recommendation system using a **Multi-Task Denoising Autoencoder (DAE)**. It learns from implicit (presence) and explicit (rating) feedback.

**Dataset**: [Anime Dataset (Jan 1917 to Oct 2025)](https://www.kaggle.com/datasets/neelagiriaditya/anime-dataset-jan-1917-to-oct-2025)
**Framework**: JAX + Flax

Based on learnings from the Sprout repository, this model features:
- Dual-head decoder (Presence prediction + Rating regression)
- Uncertainty weighting for multi-task loss balance
- Hybrid absolute/relative rating normalization

In [ ]:
import os
import pandas as pd
import numpy as np
import jax
import jax.numpy as jnp
from jax import random
import flax.linen as nn
from flax.training import train_state
import optax
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading and Preprocessing
We will load `ratings.csv` and `details.csv` from the Kaggle dataset. We filter out rare items to create a dense corpus of the top N most popular anime, and filter out users with too few ratings.

In [ ]:
# config
CORPUS_SIZE = 6000
MIN_USER_RATINGS = 10
BATCH_SIZE = 512
HIDDEN_DIM = 2048
BOTTLENECK_DIM = 512
LEARNING_RATE = 3e-4
DROPOUT_RATE = 0.4
EPOCHS = 10

DATA_DIR = '/kaggle/input/datasets/neelagiriaditya/anime-dataset-jan-1917-to-oct-2025/datasets'
RATINGS_PATH = os.path.join(DATA_DIR, 'ratings.csv')
DETAILS_PATH = os.path.join(DATA_DIR, 'details.csv')

In [ ]:
print("Loading datasets...")

ratings_df = pd.read_csv(RATINGS_PATH, usecols=['username', 'mal_id', 'score', 'status'])

# Determine top anime for the corpus
anime_counts = ratings_df['mal_id'].value_counts()
corpus_ids = anime_counts.head(CORPUS_SIZE).index.values
corpus_id_to_idx = {aid: idx for idx, aid in enumerate(corpus_ids)}

# Filter ratings to only include corpus items
ratings_df = ratings_df[ratings_df['mal_id'].isin(corpus_id_to_idx)]

# Filter users with less than MIN_USER_RATINGS
user_counts = ratings_df['username'].value_counts()
valid_users = user_counts[user_counts >= MIN_USER_RATINGS].index
ratings_df = ratings_df[ratings_df['username'].isin(valid_users)]

print(f"Retained {len(valid_users)} users and {len(ratings_df)} ratings.")
'\n',

# Save the corpus mapping for backend use\n
import json
with open('/kaggle/working/corpus_mapping.json', 'w') as f:
    json.dump({'corpus_ids': corpus_ids.tolist()}, f)
print("Saved corpus mapping to /kaggle/working/corpus_mapping.json")

## 2. Rating Normalization
We use a hybrid approach that mixes Z-Score normalization (relative user preference) and Absolute Scale normalization. Dropped unrated items act as a strong negative signal.

In [ ]:
def normalize_user_ratings(scores, statuses):
    """
    Normalizes a single user's ratings blending relative and absolute scales.
    """
    scores = np.asarray(scores, dtype=np.float32).copy()
    statuses = np.asarray(statuses)
    
    # Handle dropped unrated items with a sentinel -2
    mask_dropped_unrated = (statuses == 'dropped') & (scores == 0)
    scores[mask_dropped_unrated] = -2
    
    valid_scores = scores[scores > 0]
    if len(valid_scores) > 1:
        mu = np.mean(valid_scores)
        sigma = np.std(valid_scores) + 1e-6
    else:
        mu = 5.0
        sigma = 2.0
        
    # Impute unrated watched items with the mean
    scores[(scores == 0) & (statuses != 'dropped')] = mu
    # Assign low score to dropped unrated items (1.5 std below mean)
    scores[scores == -2] = mu - 1.5 * sigma
    
    # Z-Score (relative)
    z_score = np.clip((scores - mu) / sigma, -3.0, 3.0)
    # Absolute scale (neutral point 5.5)
    abs_score = np.clip((scores - 5.5) / 2.5, -2.5, 2.0)
    
    # Mix based on variance (users with low variance get more absolute scaling)
    alpha = np.clip(sigma / 2.6, 0.3, 0.8)
    return np.clip(alpha * z_score + (1 - alpha) * abs_score, -2.5, 2.5)

print("Normalizing ratings (this may take a bit for a large dataset)...")
user_groups = ratings_df.groupby('username')
all_vectors = []
all_masks = []

for username, group in tqdm(user_groups, desc="Normalizing Profiles"):
    idxs = group['mal_id'].map(corpus_id_to_idx).values
    scores = group['score'].values
    statuses = group['status'].values
    
    norm_scores = normalize_user_ratings(scores, statuses)
    rated_mask = scores > 0 # True if actually rated
    
    # Build dense vector: [Presence (size N), Ratings (size N)]
    vec = np.zeros(CORPUS_SIZE * 2, dtype=np.float32)
    rmask = np.zeros(CORPUS_SIZE, dtype=np.float32)
    
    vec[idxs] = 1.0 # Presence
    vec[CORPUS_SIZE + idxs] = norm_scores # Ratings
    rmask[idxs] = rated_mask.astype(np.float32)
    
    all_vectors.append(vec)
    all_masks.append(rmask)

X_train = np.stack(all_vectors)
M_train = np.stack(all_masks)
print(f"Training data shape: {X_train.shape}")

## 3. Model Architecture (JAX / Flax)
We define a Multi-Task Denoising Autoencoder. The encoder compresses the user profile into a bottleneck, and the dual-head decoder predicts presence (implicit feedback) and ratings (explicit feedback).

In [ ]:
class Recommender(nn.Module):
    hidden_dim: int = HIDDEN_DIM
    bottleneck_dim: int = BOTTLENECK_DIM
    corpus_size: int = CORPUS_SIZE

    @nn.compact
    def __call__(self, x, training: bool = False):
        # --- Encoder ---
        h = nn.Dense(self.hidden_dim)(x)
        h = nn.swish(h)
        bottleneck = nn.Dense(self.bottleneck_dim, name="bottleneck")(h)
        
        if training:
            # Add noise to latent space for regularization
            rng = self.make_rng('noise')
            noise = random.normal(rng, bottleneck.shape) * 0.1
            z = bottleneck + noise
        else:
            z = bottleneck
            
        # --- Decoder Head 1: Item Presence (Logits) ---
        d1 = nn.Dense(self.hidden_dim // 2)(z)
        d1 = nn.swish(d1)
        d1 = nn.Dense(self.hidden_dim)(d1)
        d1 = nn.swish(d1)
        item_logits = nn.Dense(self.corpus_size, name="item_logits")(d1)
        
        # --- Decoder Head 2: Rating Prediction ---
        d2 = nn.Dense(self.hidden_dim // 2)(z)
        d2 = nn.swish(d2)
        d2 = nn.Dense(self.hidden_dim)(d2)
        d2 = nn.swish(d2)
        rating_pred = nn.Dense(self.corpus_size, name="rating_pred")(d2)
        
        # Learnable uncertainty weights for multi-task loss
        log_var_presence = self.param('log_var_presence', nn.initializers.zeros, (1,))
        log_var_rating = self.param('log_var_rating', nn.initializers.zeros, (1,))
        
        return item_logits, rating_pred, log_var_presence, log_var_rating

## 4. Training Loop
We train the model using Multinomial Log-Likelihood for the presence head and masked Huber loss for the rating head. Input profiles are dynamically corrupted (dropout) during training.

In [ ]:
class TrainState(train_state.TrainState):
    key: jax.Array

def create_train_state(rng, learning_rate):
    model = Recommender()
    dummy_input = jnp.ones((1, CORPUS_SIZE * 2))
    params = model.init({"params": rng, "noise": rng}, dummy_input)["params"]
    tx = optax.adam(learning_rate)
    return TrainState.create(apply_fn=model.apply, params=params, tx=tx, key=rng)

@jax.jit
def train_step(state, batch, rated_mask):
    presence = batch[:, :CORPUS_SIZE]
    ratings_z = batch[:, CORPUS_SIZE:]
    
    dropout_rng, vae_rng = random.split(state.key)
    
    # Dynamic dropout to act as denoising autoencoder
    keep = random.bernoulli(dropout_rng, p=(1.0 - DROPOUT_RATE), shape=presence.shape)
    corrupted_presence = presence * keep
    corrupted_ratings = ratings_z * keep
    x_in = jnp.concatenate([corrupted_presence, corrupted_ratings], axis=1)
    
    def loss_fn(params):
        item_logits, rating_pred, log_var_presence, log_var_rating = state.apply_fn(
            {"params": params}, x_in, training=True, rngs={"noise": vae_rng}
        )
        
        # Presence Multinomial Loss
        log_probs = jax.nn.log_softmax(item_logits, axis=1)
        per_user_counts = jnp.maximum(jnp.sum(presence, axis=1), 1.0)
        presence_loss = jnp.mean(-jnp.sum(presence * log_probs, axis=1) / per_user_counts)
        
        # Rating Huber Loss (masked to actually rated items)
        err = rating_pred - ratings_z
        per_entry_loss = optax.huber_loss(err, delta=1.0)
        denom_r = jnp.maximum(jnp.sum(rated_mask, axis=1), 1.0)
        rating_loss = jnp.mean(jnp.sum(rated_mask * per_entry_loss, axis=1) / denom_r)
        
        # Uncertainty Weighting
        precision_p = jnp.exp(-log_var_presence)
        precision_r = jnp.exp(-log_var_rating)
        weighted_loss = (precision_p * presence_loss + log_var_presence) + \
                        (precision_r * rating_loss + log_var_rating)
        
        return jnp.mean(weighted_loss), (presence_loss, rating_loss)
        
    (loss, (recon_p, recon_r)), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
    state = state.apply_gradients(grads=grads)
    state = state.replace(key=dropout_rng)
    return state, loss, recon_p, recon_r

# Data Generator
def data_generator(X, M, batch_size):
    num_samples = X.shape[0]
    indices = np.arange(num_samples)
    np.random.shuffle(indices)
    for i in range(0, num_samples, batch_size):
        batch_idx = indices[i:i+batch_size]
        yield X[batch_idx], M[batch_idx]

# Initialization & Training
rng = random.PRNGKey(42)
state = create_train_state(rng, LEARNING_RATE)

print("Starting Training...")
for epoch in range(EPOCHS):
    gen = data_generator(X_train, M_train, BATCH_SIZE)
    epoch_losses = []
    num_batches = int(np.ceil(X_train.shape[0] / BATCH_SIZE))
    for batch_x, batch_m in tqdm(gen, total=num_batches, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        state, loss, p_loss, r_loss = train_step(state, jnp.array(batch_x), jnp.array(batch_m))
        epoch_losses.append(loss)
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {np.mean(epoch_losses):.4f}")

# Save model weights for backend use
from flax import serialization
with open('/kaggle/working/model_weights.msgpack', 'wb') as f:
    f.write(serialization.to_bytes(state.params))
    
print("Saved model weights to /kaggle/working/model_weights.msgpack")

## 5. Generating Recommendations
Once trained, we can perform inference. We combine the predicted item probability and rating into a final recommendation score.

In [ ]:
def get_recommendations(state, user_idx, top_k=20, logit_weight=0.3):
    user_input = X_train[user_idx:user_idx+1]
    item_logits, rating_pred, _, _ = state.apply_fn(
        {"params": state.params}, jnp.array(user_input), training=False
    )
    
    logits_1d = item_logits[0]
    preds_1d = rating_pred[0]
    presence_mask = user_input[0, :CORPUS_SIZE]
    
    # Z-Score the logits and ratings for linear mixing
    norm_logits = (logits_1d - jnp.mean(logits_1d)) / (jnp.std(logits_1d) + 1e-6)
    norm_ratings = (preds_1d - jnp.mean(preds_1d)) / (jnp.std(preds_1d) + 1e-6)
    
    combined_score = (logit_weight * norm_logits) + ((1.0 - logit_weight) * norm_ratings)
    
    # Mask out items already in the user's profile
    masked_score = jnp.where(presence_mask > 0, -jnp.inf, combined_score)
    
    # Get top K
    top_indices = jnp.argsort(masked_score)[-top_k:][::-1]
    top_scores = masked_score[top_indices]
    
    # Map corpus indices back to MAL IDs
    mal_ids = [corpus_ids[i] for i in top_indices]
    return mal_ids, top_scores

print("Sample Recommendations for User 0:")
recs, scores = get_recommendations(state, user_idx=0)
for rank, (aid, score) in enumerate(zip(recs, scores), 1):
    print(f"{rank}. MAL ID: {aid} (Score: {score:.3f})")